# PoliWAM - Difficulty Assignment

Assigns `difficulty` (Easy / Medium / Hard) to 200 randomly sampled PoliWAM rows.

**Method.** Three models are each given the same strict multiple-choice quiz -
*which category does this message belong to?* - where the options are the
distinct `answer` labels present in the 200 sampled rows. Each model scores 1
(correct) or 0 (wrong), and the three are summed:

| Correct | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |


**Weights: download once, reuse forever.** Models are fetched from HuggingFace,
quantised to 4-bit, and then **saved to your Drive**. Every later run loads the
4-bit copy straight from Drive - no download, no re-quantisation. The first run
is slow; subsequent runs are not.

**Crash-safe.** Every batch of scored rows is appended to a per-model progress
file the moment it finishes, so a Colab disconnect never loses completed work.
Re-run and it resumes from the last saved batch.

**Output.** `poliwam_difficulty.jsonl` - the original 14 schema fields with
`difficulty` filled in, plus `poliwam_audit.jsonl` holding each model's vote.



### Cell 1 - Install dependencies and authenticate

Installs the HuggingFace stack, then logs in.

**An HF token is required.** Llama-3.1 and Gemma-2 are gated models - only
Mistral is open. To get access:

1. Open [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
   and [gemma-2-9b-it](https://huggingface.co/google/gemma-2-9b-it) and accept
   each license (one click; approval is usually instant).
2. Create a **read** token at
   [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. In Colab, click the **key icon** in the left sidebar, add a secret named
   `HF_TOKEN`, paste the token, and enable notebook access.

Use the secret rather than pasting the token into a cell - a pasted token gets
saved inside the notebook file. If the token is missing, Mistral still works
and the other two fail with a 401 at load time.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

# ---- HuggingFace login (needed for gated Llama / Gemma) ----
HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")
    print("Add HF_TOKEN via the key icon in the left sidebar, then re-run.")

### Cell 2 - Mount Drive

Drive is the weight cache. Mounting is treated as a hard requirement and
reported honestly: `DRIVE_OK` records whether it actually worked, and later
cells check it rather than assuming.

If you see **"credential propagation was unsuccessful"**, the auth popup did
not complete. Fixes, in order:

1. Re-run this cell and finish the Google popup fully (pick account -> allow).
2. Allow pop-ups and third-party cookies for `colab.research.google.com`.
3. Mount manually: Files sidebar (folder icon) -> "Mount Drive".
4. Runtime -> Restart session, then re-run.

Without Drive the notebook still runs, but nothing is cached, so every session
re-downloads ~49 GB. Caching needs roughly **15 GB free** on Drive - a free
account has 15 GB total, so check your space before the first run.

In [ ]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
    try:
        st = os.statvfs(DRIVE_MOUNT)
        print("free on Drive: {:.1f} GB".format(st.f_bavail * st.f_frsize / 1e9))
    except Exception:
        pass
else:
    print("\nWARNING: no Drive. The notebook will still run, but weights are")
    print("NOT cached - every session re-downloads ~49 GB. See the notes above.")

### Cell 3 - Configuration

All tunable settings.

- `N_ROWS` / `SEED` - size and reproducibility of the random sample.
- `BATCH_SIZE` - rows scored before results are flushed to disk. Smaller means
  less work lost to a disconnect.
- `USE_INSTRUCT` - `True` picks the instruction-tuned checkpoints, which are
  clearly better at a classification quiz. Set `False` for the base models;
  Cell 5 detects which you have and switches prompt style automatically, so
  either works.
- `MODELS` - three judges from three **different families** (Mistral / Meta /
  Google). Family diversity matters more than raw accuracy here: judges from
  one family share training data and make *correlated* mistakes, which pushes
  every row to 0/3 or 3/3 and empties the Medium band that carries the most
  signal.
- `gemma` sets `attn: "eager"`. Gemma-2 uses attention logit soft-capping that
  the default SDPA path does not implement - and since this notebook reads raw
  logits, that would quietly corrupt its answers. Its 8K context is the
  shortest of the three but irrelevant: prompts here are under 1K tokens.
- `bnb_config` uses `float16` because the T4 has no bfloat16 support. It is
  applied only when quantising a fresh download; a checkpoint restored from the
  Drive cache already carries its own `quantization_config`.

In [ ]:
import gc
import json
import random
import shutil
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- paths ----
INPUT_FILE  = "poliwam.jsonl"            # upload to Colab, or copy from Drive
OUTPUT_FILE = "poliwam_difficulty.jsonl"
AUDIT_FILE  = "poliwam_audit.jsonl"
PROG_DIR    = "judge_progress"           # one progress file per model

# ---- sampling ----
N_ROWS     = 200
SEED       = 42
STRATIFIED = False        # True -> roughly equal rows per answer label

# ---- batching ----
BATCH_SIZE = 25           # rows per save point

# ---- the three judges ----
USE_INSTRUCT = True

REPOS = {
    True: {                                   # instruction-tuned
        "mistral": "mistralai/Mistral-7B-Instruct-v0.3",   # ungated
        "llama":   "meta-llama/Llama-3.1-8B-Instruct",     # GATED
        "gemma":   "google/gemma-2-9b-it",                 # GATED
    },
    False: {                                  # base
        "mistral": "mistralai/Mistral-7B-v0.3",            # ungated
        "llama":   "meta-llama/Llama-3.1-8B",              # GATED
        "gemma":   "google/gemma-2-9b",                    # GATED
    },
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# the 14 IndicSample fields, in schema order
SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)

print("Judges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached in Drive" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

### Cell 4 - Load the dataset and sample 200 rows

Reads all 2000 PoliWAM rows and draws 200 at random under a fixed seed, so the
same 200 come back on every re-run - which is what makes the resume logic safe
across sessions.

The quiz options are built **from the sample itself**: every distinct `answer`
value in those 200 rows becomes one lettered choice, sorted alphabetically so
the letter mapping is identical on every run. That last part matters, because
progress files persist across sessions and a shifting mapping would silently
corrupt a resume.

Because options come from the sample rather than a hardcoded list, no option
can appear that no row is labelled with, and no gold label can be missing from
the options. Change `SEED`, `N_ROWS` or `STRATIFIED` and they rebuild.

Read the printed distribution before continuing: rare combined labels such as
`Spam,Others` may appear only once or twice.

In [ ]:
with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))

random.seed(SEED)
if STRATIFIED:
    by_label = {}
    for r in all_rows:
        by_label.setdefault(r["answer"], []).append(r)
    per = N_ROWS // len(by_label)
    sample = []
    for label, group in by_label.items():
        random.shuffle(group)
        sample += group[:per]
    random.shuffle(sample)
    sample = sample[:N_ROWS]
else:
    sample = random.sample(all_rows, N_ROWS)

# options = every distinct answer label present in the sampled rows.
# sorted() keeps the letter mapping stable across runs and machines.
OPTIONS = sorted({r["answer"] for r in sample})
LETTERS = [chr(ord("A") + i) for i in range(len(OPTIONS))]
LET_OF  = {opt: let for opt, let in zip(OPTIONS, LETTERS)}

gold_counts = Counter(r["answer"] for r in sample)
print("Sampled {} rows | {} options built from the sample".format(
    len(sample), len(OPTIONS)))
for o in OPTIONS:
    print("  {}. {:<30} n={}".format(LET_OF[o], o, gold_counts[o]))

maj = gold_counts.most_common(1)[0]
print("\nMajority-class baseline: {:.1%} ('{}')".format(maj[1] / len(sample), maj[0]))

### Cell 5 - Build the quiz prompt

Two prompt styles are defined, and Cell 6 picks per checkpoint:

- `build_completion` - a flat few-shot prompt for **base** models, which do not
  follow instructions but do continue patterns. Worked examples in a rigid
  `Message: ... / Answer: X` format, ending with the row to score and a
  dangling `Answer:` for the model to complete.
- `build_chat_messages` - the same content as chat turns for **instruct**
  models, with the category definitions in a system message.

**Few-shot examples are drawn from outside the 200-row sample.** If they came
from the sample, the gold answers for those rows would be handed to the model
and their difficulty would be meaningless. One example per option label, picked
deterministically from `SEED`, so every model and every re-run sees the same
examples.

Newlines inside a message are flattened to spaces so a multi-line WhatsApp
forward cannot break the `Message:` / `Answer:` pattern.

The `explanation` field (`favour=`, `against=`, `isPolitical=`) is never shown -
it leaks the answer.

In [ ]:
INSTRUCTIONS = (
    "Messages below are taken from Indian WhatsApp political discussion "
    "groups. They may be in Hindi, English or code-mixed Hinglish, written in "
    "Devanagari or Roman script. Each message is labelled with one category:\n"
    "- Spam: forwarded chain messages, greetings, devotional or festival "
    "forwards, links, filler with no informational value to the group\n"
    "- Offensive: insults, abuse, slurs, mockery or derogatory attacks on "
    "people, parties, communities or religions\n"
    "- Advertisement: promotion of a product, service, business, app, job "
    "or event\n"
    "- Others: ordinary political commentary, news, opinion or discussion "
    "that fits none of the above"
)


def options_block():
    return "\n".join("{}. {}".format(LET_OF[o], o) for o in OPTIONS)


def flat(text):
    # keep the Message:/Answer: pattern unambiguous for a base model
    return " ".join(text.split())


def pick_fewshot():
    # one worked example per option label, taken from rows OUTSIDE the sample
    # so no scored row ever has its gold answer shown to the model
    used = {r["id"] for r in sample}
    pool = [r for r in all_rows
            if r["id"] not in used
            and r["answer"] in OPTIONS
            and 12 <= len(r["question"]) <= 110]
    random.Random(SEED + 1).shuffle(pool)

    shots, seen = [], set()
    for r in pool:
        if r["answer"] in seen:
            continue
        shots.append(r)
        seen.add(r["answer"])
        if len(seen) == len(OPTIONS):
            break
    return shots


FEWSHOT = pick_fewshot()


def build_completion(message):
    head = INSTRUCTIONS + "\n\nCategories:\n" + options_block() + "\n"
    body = ""
    for r in FEWSHOT:
        body += "\nMessage: {}\nAnswer: {}\n".format(
            flat(r["question"]), LET_OF[r["answer"]])
    body += "\nMessage: {}\nAnswer:".format(flat(message))
    return head + body


def build_chat_messages(message):
    query = ("Message:\n{}\n\nWhich category does this message belong to?\n"
             "Options:\n{}\n\nAnswer with one letter ({}):")
    msgs = [{"role": "system",
             "content": INSTRUCTIONS + "\n\nAnswer with a SINGLE letter. "
                        "Output only that letter."}]
    for r in FEWSHOT:
        msgs.append({"role": "user", "content": query.format(
            flat(r["question"]), options_block(), "/".join(LETTERS))})
        msgs.append({"role": "assistant", "content": LET_OF[r["answer"]]})
    msgs.append({"role": "user", "content": query.format(
        flat(message), options_block(), "/".join(LETTERS))})
    return msgs


print("Few-shot examples ({}), all from OUTSIDE the 200-row sample:".format(
    len(FEWSHOT)))
for r in FEWSHOT:
    print("  {} -> {}. {}".format(r["id"], LET_OF[r["answer"]], r["answer"]))

print("\n" + "=" * 62)
print(build_completion(sample[0]["question"]))
print("=" * 62)
print("[gold answer: {}. {}]".format(
    LET_OF[sample[0]["answer"]], sample[0]["answer"]))

### Cell 6 - Strictly constrained answering

`choose_option` runs **one forward pass** and reads the logits at the final
position, then takes the argmax over just the option letters' token ids.

Two consequences:

- The model *cannot* answer off-list. The constraint is structural, so there is
  no output parsing, no retries, and no unparseable rows. That matters most for
  base models, which would otherwise ramble instead of answering.
- It is far faster than generating text - 200 rows take about a minute per
  model rather than several.

`format_prompt` picks the style per checkpoint: no `chat_template` means a base
model and the flat completion prompt; a template means chat turns, with a
fallback that folds the system prompt into the first user turn for templates
that reject a `system` role, as Gemma's does.

`letter_token_ids` collects each letter's first token both bare (`A`) and
space-prefixed (` A`). That is load-bearing: after `Answer:` a base model
predicts ` A` with the leading space, while a chat template usually starts a
fresh line and predicts bare `A`.

In [ ]:
def letter_token_ids(tokenizer):
    ids = {}
    for let in LETTERS:
        variants = set()
        for form in (let, " " + let):
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                variants.add(enc[0])
        ids[let] = sorted(variants)
    return ids


def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def format_prompt(tokenizer, message):
    if prompt_style(tokenizer) == "completion":
        return build_completion(message)

    msgs = build_chat_messages(message)
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # some templates (Gemma) reject a system role - fold it into the first
        # user turn rather than dropping the category definitions
        merged = [dict(m) for m in msgs[1:]]
        merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
        return tokenizer.apply_chat_template(
            merged, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def choose_option(model, tokenizer, tok_ids, message):
    text   = format_prompt(tokenizer, message)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    logits = model(**inputs).logits[0, -1]
    best = max(LETTERS, key=lambda let: max(logits[i].item() for i in tok_ids[let]))
    return OPTIONS[LETTERS.index(best)]


def clear_hf_cache():
    # the fp16 download is no longer needed once the 4-bit copy is saved
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()

print("Answering functions defined")

### Cell 7 - Load-or-cache, and the batched runner

`load_model` implements the download-once rule:

1. If `models/<name>_4bit` exists in Drive, load that. It already carries its
   own `quantization_config`, so passing a fresh `BitsAndBytesConfig` would
   conflict - it is deliberately omitted.
2. Otherwise download the fp16 repo from HuggingFace, quantise to 4-bit on the
   way in, then `save_pretrained` the 4-bit copy plus tokenizer to Drive. The
   next run takes path 1.

`trust_remote_code` stays **off**. Repo-shipped modelling code is often written
against an older transformers API - Phi-3.5, for instance, crashes with
`DynamicCache has no attribute from_legacy_cache`. All three architectures here
are native to transformers.

`run_model` is the crash-safety core. It skips rows already in
`judge_progress/<model>.jsonl`, scores the rest in batches of `BATCH_SIZE`, and
**appends each finished batch before starting the next**. At most `BATCH_SIZE`
rows are ever repeated after a disconnect, and a fully-scored model is skipped
without loading weights at all.

In [ ]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached   = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source   = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"      # settings baked into the ckpt
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time, a few minutes)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs will skip the download")

    style = prompt_style(tokenizer)
    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9,
        "few-shot completion (base)" if style == "completion"
        else "chat template (instruct)"))
    return model, tokenizer


def run_model(spec, rows):
    prog_file = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    # ---- 1. resume from whatever is already on disk ----
    done = {}
    if os.path.exists(prog_file):
        with open(prog_file, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already scored".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    # ---- 2. load (from Drive if cached, else download and cache) ----
    model, tokenizer = load_model(spec)
    tok_ids = letter_token_ids(tokenizer)

    # ---- 3. score in batches, saving after each one ----
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_start in range(0, len(remaining), BATCH_SIZE):
        batch     = remaining[batch_start : batch_start + BATCH_SIZE]
        batch_num = batch_start // BATCH_SIZE + 1

        batch_results = []
        for row in batch:
            predicted = choose_option(model, tokenizer, tok_ids, row["question"])
            batch_results.append({
                "id":        row["id"],
                "predicted": predicted,
                "gold":      row["answer"],
                "correct":   int(predicted == row["answer"]),
            })

        with open(prog_file, "a", encoding="utf-8") as f:
            for item in batch_results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        done.update({item["id"]: item for item in batch_results})
        acc = sum(v["correct"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | running acc {:.1%}".format(
            batch_num, total_batches, len(done), len(rows), acc))

    # ---- 4. free VRAM and disk for the next model ----
    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

### Cell 8 - Run all three models

Runs the judges **one at a time** - loaded, scored, unloaded - so peak VRAM
stays around 6 GB instead of the ~17 GB all three would need together. The fp16
download cache is wiped after each model so Colab's disk does not fill.

This is the long cell. On the first run it downloads ~49 GB and writes ~15 GB
of 4-bit weights to Drive; afterwards it just reads the cache. Safe to re-run -
anything already scored is skipped.

In [ ]:
preds = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    preds[spec["name"]] = run_model(spec, sample)

print("\nAll models done")

### Cell 9 - Assign difficulty into the schema

Sums the three 1/0 scores per row, maps the total to Easy / Medium / Hard, and
writes it into the row's `difficulty` field.

Each output row is rebuilt key-by-key from `SCHEMA_KEYS`, so the file carries
**exactly the 14 IndicSample fields in schema order** - `difficulty` is the
only value that changes and no extra keys leak in. Per-model votes go to the
separate audit file, keeping the dataset clean.

In [ ]:
def get_difficulty(pattern):
    score = sum(pattern)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results = []
audit = []

for row in sample:
    scores     = [preds[s["name"]][row["id"]]["correct"] for s in MODELS]
    difficulty = get_difficulty(scores)

    enriched = {**row, "difficulty": difficulty}
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":          row["id"],
        "difficulty":  difficulty,
        "scores":      scores,
        "gold_answer": row["answer"],
        "predictions": {s["name"]: preds[s["name"]][row["id"]]["predicted"]
                        for s in MODELS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))

### Cell 10 - Verify and report

Four checks before trusting the file:

1. **Schema** - all 14 keys in order, no nulls left in `difficulty`.
2. **Difficulty distribution** - the Easy / Medium / Hard split.
3. **Per-model accuracy vs. the majority-class baseline.** This is the one that
   matters. `Others` is roughly 47% of the sample, so a model answering
   `Others` every time scores ~47% while classifying nothing. With 5 options,
   random guessing is 20% - anything at or below that is broken, not weak.
4. **Prediction spread per model** - if a model's predictions pile onto one
   label, its votes are noise, not signal.

If a model looks degenerate, open `poliwam_audit.jsonl`, confirm, then either
replace it in Cell 3 or set `STRATIFIED = True` and re-run (delete
`judge_progress/` first, since the sample changes).

Rare combined labels such as `Spam,Others` may appear in only one or two rows.
No model is likely to pick them, so those rows land in Hard by default rather
than by measured difficulty - worth excluding by hand if you care.

In [ ]:
# 1. schema integrity
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(missing)))

# 2. difficulty distribution
dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution:")
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

# 3. per-model accuracy vs. baselines
print("\nPer-model accuracy:")
for s in MODELS:
    acc = sum(v["correct"] for v in preds[s["name"]].values()) / total
    print("  {:<10} {:.1%}".format(s["name"], acc))

maj_label, maj_n = Counter(r["answer"] for r in sample).most_common(1)[0]
print("  {:<10} {:.1%}  <- majority-class baseline ('{}')".format(
    "baseline", maj_n / total, maj_label))
print("  {:<10} {:.1%}  <- random guessing over {} options".format(
    "random", 1.0 / len(OPTIONS), len(OPTIONS)))

# 4. prediction spread - catches a model that always answers the same thing
print("\nPrediction spread:")
for s in MODELS:
    c = Counter(v["predicted"] for v in preds[s["name"]].values())
    top, n = c.most_common(1)[0]
    flag = "  <- DEGENERATE, votes are noise" if n / total > 0.9 else ""
    print("  {:<10} {}{}".format(s["name"], dict(c), flag))

print("\nSample rows:")
for r in final_results[:3]:
    print("  {} | {:<6} | {:<14} | {}".format(
        r["id"], r["difficulty"], r["answer"],
        " ".join(r["question"].split())[:40]))